# Differentiable simulation — x-position loss scan

Compare the differentiable simulator against the stochastic one and show that an L2 loss on the dense waveform has a clean basin around the true x.

1. jaxtpc → one event (volume 0) → voxelize at **20 mm**
2. Build two simulators sharing the **same SIREN TOF sampler**:
   - `OpticalSimulator` — stochastic (Poisson + inverse-CDF)
   - `DifferentiableOpticalSimulator` — deterministic PDF deposition
3. Side-by-side comparison at the true x (averaged stoch vs diff)
4. Scan `dx ∈ [-200, +200] mm`, computing `L2(diff_sim(x+dx), stoch_target)`

In [ ]:
import os, sys, gc, time
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
sys.path.insert(0, '..')
sys.path.insert(0, '../jaxtpc')

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import SymLogNorm

import jax
import jax.numpy as jnp

from tools.geometry import generate_detector
from tools.simulation import DetectorSimulator
from tools.loader import load_event

from goop import (
    DifferentiableOpticalSimulator,
    OpticalSimConfig,
    OpticalSimulator,
    Response,
    SERKernel,
    create_siren_tof_sampler,
    voxelize,
)
from goop.delays import Delays

torch.manual_seed(0)
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('torch device:', DEVICE, '| jax devices:', jax.devices())

## 1. Load + voxelize one event

In [ ]:
DATA_PATH   = '/sdf/home/y/youngsam/sw/dune/sirentv/data/out.h5'
CONFIG_PATH = '../jaxtpc/config/cubic_wireplane_config.yaml'
EVENT_IDX   = 0
VOXEL_DX    = 20.0  # mm

detector_config = generate_detector(CONFIG_PATH)
jx_sim = DetectorSimulator(
    detector_config,
    total_pad=250_000,
    response_chunk_size=50_000,
    include_track_hits=False,
)
jx_cfg = jx_sim.config
_warm = jx_sim.process_event_light(load_event(DATA_PATH, jx_cfg, event_idx=0))
jax.block_until_ready(_warm.volumes[0].charge); del _warm

deposits = load_event(DATA_PATH, jx_cfg, event_idx=EVENT_IDX)
filled   = jx_sim.process_event_light(deposits)
jax.block_until_ready(filled.volumes[0].charge)

vol = filled.volumes[0]
n_raw = int(vol.n_actual)
pos_raw = np.asarray(vol.positions_mm[:n_raw], dtype=np.float32)
nph_raw = np.asarray(jnp.ceil(vol.photons[:n_raw]).astype(jnp.int32))
tns_raw = np.asarray(vol.t0_us[:n_raw], dtype=np.float32) * 1000.0

del deposits, filled, vol
jax.clear_caches(); gc.collect(); torch.cuda.empty_cache()

pos_v, nph_v, tns_v = voxelize(pos_raw, nph_raw, tns_raw, dx=VOXEL_DX)
pos_base  = torch.from_numpy(pos_v).to(DEVICE)
n_photons = torch.from_numpy(nph_v.astype(np.int64)).to(DEVICE)
t_step    = torch.from_numpy(tns_v).to(DEVICE)

print(f'event {EVENT_IDX}: raw={n_raw:,}  voxels(dx={VOXEL_DX:.0f}mm)={len(pos_v):,}  '
      f'reduction={n_raw/len(pos_v):.1f}x')
print(f'  photon yield: {int(n_photons.sum().item()):,}  (preserved: {nph_raw.sum() == nph_v.sum()})')
print(f'  pos x: [{pos_v[:,0].min():.0f}, {pos_v[:,0].max():.0f}] mm')

## 2. Build the two simulators (shared SIREN sampler)

We reuse one `SirenTOFSampler` instance for both pipelines so any difference is purely stochastic-vs-PDF-deposition (not a different model). Delays are stripped (`Delays([])`) so the diff target equals the stoch mean modulo Poisson noise; baseline noise + SER jitter off.

In [ ]:
TICK_NS = 1.0
GAIN    = -45.0

siren = create_siren_tof_sampler(device=str(DEVICE))

def make_cfg():
    return OpticalSimConfig(
        tof_sampler=siren,
        delays=Delays([]),
        kernel=Response(
            kernels=[SERKernel(duration_ns=2000.0, device=DEVICE)],
            tick_ns=TICK_NS, device=DEVICE,
        ),
        device=str(DEVICE),
        tick_ns=TICK_NS,
        gain=GAIN,
        ser_jitter_std=0.0,
        baseline_noise_std=0.0,
        # voxelized N is small; per-chunk checkpointing is unnecessary overhead.
        stream_checkpoint=False,
    )

sim_stoch = OpticalSimulator(make_cfg())
sim_diff  = DifferentiableOpticalSimulator(make_cfg())
print(f'stoch + diff sims ready, n_channels = {sim_stoch.config.n_channels}')

## 3. Build the stochastic target

Average `N_DRAWS` independent stochastic runs at the true x to suppress Poisson shot noise; this is the "truth" the diff sim is being compared against. Single-run kept aside for visualization.

In [ ]:
N_DRAWS = 10

stoch_runs = []
for i in range(N_DRAWS):
    sw = sim_stoch.simulate(pos_base, n_photons, t_step,
                            stitched=True, subtract_t0=True,
                            add_baseline_noise=False)
    stoch_runs.append(sw.deslice())

wf_stoch_one = stoch_runs[0]

# Place every draw on a common grid (the union of all draws), then average.
ref = stoch_runs[0]
for i in range(1, N_DRAWS):
    ref, _ = ref.align_with(stoch_runs[i])
adcs = []
for w in stoch_runs:
    adcs.append(w.align_to(ref.t0, ref.adc.shape[1]).adc)
adc_target = torch.stack(adcs).mean(dim=0)
t0_target  = ref.t0
n_bins_tg  = adc_target.shape[1]

print(f'target: t0={t0_target:.1f} ns, shape={tuple(adc_target.shape)}, '
      f'peak |adc|={adc_target.abs().max().item():.1f}')

## 4. Diff vs stoch at the true x

Run the diff sim once and compare against the averaged stoch target. If the SIREN model + PDF deposition is consistent with categorical sampling, the two should match to within shot-noise scatter.

In [ ]:
with torch.no_grad():
    sw_diff = sim_diff.simulate(pos_base, n_photons, t_step,
                                stitched=True, subtract_t0=True,
                                add_baseline_noise=False)
    wf_diff = sw_diff.deslice()

# Pad both onto the target's grid via the new helper.
adc_diff = wf_diff.align_to(t0_target, n_bins_tg).adc
adc_one  = wf_stoch_one.align_to(t0_target, n_bins_tg).adc

# Crop to peak window for plotting.
energy_per_bin = adc_target.abs().sum(0).cpu().numpy()
peak_bin = int(energy_per_bin.argmax())
PLOT_HALF = 1500
lo = max(0, peak_bin - PLOT_HALF); hi = min(n_bins_tg, peak_bin + PLOT_HALF)
t_ax = t0_target + np.arange(lo, hi) * TICK_NS

ch_bright = int(adc_target.abs().sum(1).argmax().item())
vmax = float(max(adc_target.abs().max().item(), adc_diff.abs().max().item()))

fig, axes = plt.subplots(2, 2, figsize=(15, 8))
for ax, arr, title in zip(
    axes[0],
    [adc_target, adc_diff],
    [f'stoch target (mean of {N_DRAWS})', 'diff sim'],
):
    im = ax.imshow(
        arr[:, lo:hi].cpu().numpy(), aspect='auto', cmap='RdBu_r',
        norm=SymLogNorm(linthresh=1.0, linscale=1.0, vmin=-vmax, vmax=vmax),
        extent=[t_ax[0], t_ax[-1], 0, arr.shape[0]], origin='lower', interpolation='none',
    )
    ax.axhline(81, ls='--', lw=0.5, color='k')
    ax.set_title(title); ax.set_xlabel('time (ns)'); ax.set_ylabel('PMT')
    plt.colorbar(im, ax=ax, label='ADC')

ax = axes[1, 0]
ax.plot(t_ax, adc_one [ch_bright, lo:hi].cpu().numpy(), lw=0.6, color='0.6', label='stoch (1 draw)')
ax.plot(t_ax, adc_target[ch_bright, lo:hi].cpu().numpy(), lw=1.0, color='k',         label=f'stoch mean (n={N_DRAWS})')
ax.plot(t_ax, adc_diff [ch_bright, lo:hi].cpu().numpy(), lw=1.0, color='tab:red',   label='diff', ls='--')
ax.set_xlabel('time (ns)'); ax.set_ylabel('ADC'); ax.set_title(f'PMT {ch_bright}')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

ax = axes[1, 1]
energy_target = adc_target.abs().sum(1).cpu().numpy()
energy_diff   = adc_diff.abs().sum(1).cpu().numpy()
active = energy_target > 1e-3 * energy_target.max()
lim = [0, max(energy_target.max(), energy_diff.max()) * 1.05]
ax.scatter(energy_target[active], energy_diff[active], s=14, alpha=0.7, color='tab:purple')
ax.plot(lim, lim, 'k--', lw=1)
ax.set_xlabel('stoch target Σ|ADC|'); ax.set_ylabel('diff Σ|ADC|')
ax.set_title(f'Per-PMT integrated energy (N_active = {int(active.sum())})')
ax.set_xlim(lim); ax.set_ylim(lim); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Loss scan in dx

For each shift `dx`, run the diff sim with `pos = pos_base + dx · ê_x`, pad onto the target grid, and compute the per-bin MSE against the stoch target. Includes one autograd check at the minimum to confirm the entire pipeline is differentiable end-to-end.

In [ ]:
EX = torch.tensor([1.0, 0.0, 0.0], device=DEVICE)

def diff_loss_at(dx):
    """diff sim at pos+dx*ex, MSE vs the stoch target. Differentiable in dx."""
    if not isinstance(dx, torch.Tensor):
        dx = torch.tensor(float(dx), device=DEVICE)
    pos_shift = pos_base + dx * EX
    sw = sim_diff.simulate(pos_shift, n_photons, t_step,
                           stitched=True, subtract_t0=True,
                           add_baseline_noise=False)
    wf = sw.deslice()
    adc = wf.align_to(t0_target, n_bins_tg).adc
    return (adc - adc_target).pow(2).mean()

# 21 dx values from -200 to +200 mm (inclusive). With ~5-8k voxels and SIREN
# this is ~10-30 s on a single GPU.
DX_GRID = np.linspace(-200.0, 200.0, 21)
losses = []
with torch.no_grad():
    for dx in DX_GRID:
        losses.append(diff_loss_at(dx).item())
losses = np.array(losses)

dx_min = float(DX_GRID[int(losses.argmin())])
print(f'min loss at dx = {dx_min:.0f} mm   (true dx = 0)')
print(f'loss(0) = {losses[len(DX_GRID)//2]:.3e}   loss(-200) = {losses[0]:.3e}   loss(+200) = {losses[-1]:.3e}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(DX_GRID, losses, 'o-', color='tab:blue')
ax.axvline(0.0, ls='--', color='k', lw=0.8, label='true dx')
ax.axvline(dx_min, ls=':', color='tab:red', lw=1.0, label=f'argmin = {dx_min:.0f} mm')
ax.set_xlabel('dx (mm)'); ax.set_ylabel('mean ((adc_diff - adc_target)^2)')
ax.set_title('Loss scan: diff sim shifted in x vs stoch target')
ax.set_yscale('log'); ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

## 6. Sanity check: gradient flows back to dx

Confirm the loss is actually differentiable wrt the shift parameter — comparing autograd's `dL/d(dx)` against a central-difference estimate around `dx=0`.

In [ ]:
dx0 = torch.tensor(0.0, device=DEVICE, requires_grad=True)
L = diff_loss_at(dx0)
L.backward()
g_auto = float(dx0.grad.item())

EPS = 5.0  # mm
with torch.no_grad():
    L_plus  = diff_loss_at( EPS).item()
    L_minus = diff_loss_at(-EPS).item()
g_fd = (L_plus - L_minus) / (2 * EPS)

print(f'autograd dL/d(dx)|_0  = {g_auto:+.3e}')
print(f'finite-diff (eps={EPS} mm) = {g_fd:+.3e}')
print(f'rel diff             = {abs(g_auto - g_fd) / max(abs(g_auto), abs(g_fd), 1e-30):.2%}')